# Data Cleaning & Exploratory Data Analysis on Invoices

Carter

Wide World Importers wants to know which customers and products are the most important to the business. The invoice data has been entered by a bunch of different programs over the years so it has standardization errors, business logic errors, null values and duplicates. This notebook cleans the data up so it can be used for machine learning (no nulls allowed) and then answers the executives questions.

## About the data

invoices.csv has one row per invoice line. Each row has the invoice ID, the item (StockItemID and Description), quantity, unit price, tax rate, tax amount, line profit, extended price (total paid) and the customer.

In [ ]:
python -c "import pandas, seaborn; print(pandas.__version__, seaborn.__version__)"

In [ ]:
import pandas as pd
df = pd.read_csv('invoices.csv')
df.head()

/tmp/ipykernel_3639/1993749872.py:2: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('invoices.csv')


,InvoiceLineID,InvoiceID,StockItemID,Description,PackageTypeID,Quantity,UnitPrice,TaxRate,TaxAmount,LineProfit,ExtendedPrice,LastEditedBy,LastEditedWhen,Customer
0,1,1,67.0,Ride on toy sedan car (Black) 1/12 scale,7,10,230.0,15,345.0,850.0,2645.0,7,NaN,Stuff by Stew
1,2,2,50.0,Developer joke mug - old C developers never di...,7,9,13.0,15,NaN,76.5,-999.0,7,1/1/2013 12:00,Wholesaler Plus
2,3,2,10.0,USB food flash drive - chocolate bar,7,9,32.0,15,43.2,180.0,331.2,7,NaN,Wholesaler Plus
3,4,3,114.0,Superhero action jacket (Blue) XXL,7,3,30.0,15,NaN,24.0,-999.0,7,NaN,Big Buys Retail
4,5,4,206.0,Permanent marker black 5mm nib (Black) 5mm,7,96,2.7,15,NaN,96.0,NaN,7,1/1/2013 12:00,Terry's Trinkets


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

The describe shows two business logic problems, TaxRate has a minimum of -15 and ExtendedPrice has a minimum of -999. A negative LineProfit is fine because you can sell something at a loss.

# Cleaning

## Standardization

The Customer column has the same companies spelled different ways (Stuff by Stew / Stuf by Stew / stuff by stew, Terry's Trinkets / Terry's Trinket). Fixing those so each customer only shows up once.

In [ ]:
df['Customer'].value_counts()

In [ ]:
df['Customer'] = df['Customer'].replace({
    'Stuf by Stew': 'Stuff by Stew',
    'stuff by stew': 'Stuff by Stew',
    "Terry's Trinket": "Terry's Trinkets"
})
df['Customer'].value_counts()

Now I can count unique invoices per customer. Stuff by Stew has the most.

In [ ]:
df.groupby('Customer')['InvoiceID'].nunique().sort_values(ascending=False)

## Null values

LastEditedWhen is almost completely empty (230,542 nulls out of 230,548) and none of the executives questions need dates, so dropping the whole column.

In [ ]:
df = df.drop(columns=['LastEditedWhen'])
df.isnull().sum()

StockItemID and Description go together, so if a row has the ID I can fill in the description from another row with the same ID. If both are missing the item is unknown and those rows get dropped.

In [ ]:
both_null = df['StockItemID'].isnull() & df['Description'].isnull()
both_null.sum()

In [ ]:
lookup = df.dropna(subset=['StockItemID', 'Description']).drop_duplicates('StockItemID').set_index('StockItemID')['Description']
df['Description'] = df['Description'].fillna(df['StockItemID'].map(lookup))
df = df[~(df['StockItemID'].isnull() & df['Description'].isnull())]
len(df)

## Business logic

Tax rates cant be negative, they were probably just entered with the wrong sign so I flipped them to positive.

In [ ]:
df['TaxRate'] = df['TaxRate'].abs()
df['TaxRate'].mean()

## Imputing

TaxAmount and ExtendedPrice both have a lot of nulls but they can be calculated from the other columns, so imputing them instead of dropping.

TaxAmount = UnitPrice x Quantity x TaxRate / 100

In [ ]:
df['TaxAmount'] = df['TaxAmount'].fillna(df['UnitPrice'] * df['Quantity'] * df['TaxRate'] / 100)
df['TaxAmount'].mean()

ExtendedPrice = (UnitPrice x Quantity) + TaxAmount. The -999 values are input errors so I treated them like nulls and recalculated them too.

In [ ]:
df['ExtendedPrice'] = df['ExtendedPrice'].replace(-999, None)
df['ExtendedPrice'] = df['ExtendedPrice'].fillna(df['UnitPrice'] * df['Quantity'] + df['TaxAmount'])
df['ExtendedPrice'].mean()

## Outliers

Using z-scores with a threshold of 3 on Quantity and ExtendedPrice. There are a lot of them but they look like real big orders, not errors, so I'm just noting them and leaving them in.

In [ ]:
z_qty = (df['Quantity'] - df['Quantity'].mean()) / df['Quantity'].std()
(z_qty >= 3).sum()

In [ ]:
z_price = (df['ExtendedPrice'] - df['ExtendedPrice'].mean()) / df['ExtendedPrice'].std()
(z_price >= 3).sum()

## Duplicates

Invoice IDs can repeat because one invoice has multiple lines, so checking duplicates on InvoiceID and InvoiceLineID together.

In [ ]:
df = df.drop_duplicates(subset=['InvoiceID', 'InvoiceLineID'])
len(df)

# Analysis

## Which item has the highest total quantity sold?

Black and orange fragile despatch tape 48mmx75m, over 207,000 units.

In [ ]:
df.groupby('Description')['Quantity'].sum().sort_values(ascending=False)

## Which item has generated the most total profit?

20 mm Double sided bubble wrap 50m, about $5.3 million.

In [ ]:
df.groupby('Description')['LineProfit'].sum().sort_values(ascending=False)

## Which item is purchased in the greatest quantity per order, on average?

The same despatch tape, about 200 per order.

In [ ]:
df.groupby('Description')['Quantity'].mean().sort_values(ascending=False)

,Quantity
Description,
Black and orange fragile despatch tape 48mmx75m,199.511450
Black and orange fragile despatch tape 48mmx100m,198.755374
Clear packaging tape 48mmx75m,145.377616
3 kg Courier post bag (White) 300x190x95mm,143.612132
Shipping carton (Brown) 356x356x279mm,141.852368
...,...
USB food flash drive - hamburger,5.322002
Superhero action jacket (Blue) L,5.308140
DBA joke mug - SELECT caffeine FROM mug (Black),5.290508


# Conclusion

After cleaning, Stuff by Stew is the biggest customer with about 31,000 invoices. Packaging supplies dominate sales, the fragile despatch tape is bought the most and in the biggest orders, but bubble wrap is where the profit is. The cleaning removed the empty date column, dropped 461 rows with unknown items, fixed the customer name typos and negative tax rates, recalculated the missing tax and price values from the formulas, and removed about 2,300 duplicate lines.